# Day 2 — LoRA / QLoRA Basics + Your First Fine-Tune

---

Today you'll train a real language model. On a free Colab GPU. In about 20 minutes.

**Key insight:** we don't retrain all the weights. We use **LoRA** (Low-Rank Adaptation) — train tiny "patch" matrices on top of the frozen base model. Result: 10–100× less GPU RAM and disk than full fine-tuning, ~same quality for style/format tasks.

**Recommended environment: Google Colab.** Runtime → Change runtime type → T4 GPU. All the setup below assumes that.


## 1. LoRA in one paragraph

The base model has huge weight matrices `W`. Instead of updating `W`, LoRA freezes it and learns a **low-rank update** `A × B` where `A` and `B` are small (rank ~8–64). At inference, the effective weight is `W + A×B`.

You end up training ~0.1–1% of the parameters. The savings are enormous.

**QLoRA** = LoRA on top of a **4-bit quantized** base model. Even less RAM. Standard practice in 2026.

You don't need to understand the linear algebra to use it. `peft` and `unsloth` handle it in 3 lines.


## 2. Recommended setup — Unsloth on Colab

**Unsloth** is a 2024/2025 library that wraps Hugging Face fine-tuning with 2× speed and less memory. It's what most new fine-tuning code uses in 2026.


In [ ]:
# Run in a Colab cell with T4 GPU
!pip install unsloth trl datasets --quiet


## 3. Load a small base model


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",   # ~6 GB in 4-bit
    max_seq_length = 2048,
    load_in_4bit = True,
)


**Why Llama 3.2 3B?**

- Small enough to fit on a free T4 GPU
- Instruct-tuned (already understands chat format)
- Meta license allows fine-tuning + commercial use
- Modern (2025) architecture; results transfer to the 8B and 70B siblings

If you want more quality and don't mind waiting: swap to `unsloth/Llama-3.1-8B-Instruct`.


## 4. Attach LoRA adapters


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                              # rank - higher = more capacity, more RAM
    lora_alpha = 16,                     # a scaling factor - keep = r
    lora_dropout = 0.0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)


**What to remember:** `r=16` is a good starting rank for most tasks. Bigger tasks / more data → try 32 or 64. Smaller = cheaper but might underfit.


## 5. Load your dataset

Upload yesterday's `triage_train.jsonl` to your Colab session (or generate more data — 6 examples is way too small for real training).


In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="triage_train.jsonl", split="train")

# Convert chat messages to the model's expected text format
def format_row(row):
    text = tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

ds = ds.map(format_row)
print(ds[0]["text"])


`apply_chat_template` handles the model-specific special tokens (`<|start_header_id|>` etc for Llama 3). Never format the strings by hand — every model family is different.


## 6. Train


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        output_dir = "outputs",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_steps = 5,
        logging_steps = 5,
        save_strategy = "no",
        report_to = "none",
        fp16 = True,
    ),
)

trainer.train()


**Numbers to know (starter defaults):**

- `learning_rate = 2e-4` — LoRA takes a higher LR than full fine-tuning.
- `num_train_epochs = 3` — usually enough for small style/format datasets.
- `batch_size × grad_accum = 8` — the "effective" batch size the model sees.
- If train loss doesn't drop: try more epochs, or check your data is clean.


## 7. Test the trained model


In [ ]:
FastLanguageModel.for_inference(model)

prompts = [
    "My subscription didn't renew properly.",
    "The mobile app keeps crashing on Android 14.",
    "Please add SSO support.",
]

for p in prompts:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": "You are a support triage bot. Classify each ticket into exactly one of: billing, technical, feature-request, other. Respond with only the label."},
         {"role": "user",   "content": p}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")

    out = model.generate(inputs, max_new_tokens=10, do_sample=False)
    print(f"{p:60}  ->  {tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()}")


**Expected:** the model outputs one of the four labels, no chit-chat. If the base model rambles ("This sounds like it could be a billing issue... let me help you..."), your fine-tune worked.


## 8. Save just the adapter (not the whole model)


In [ ]:
model.save_pretrained("triage-lora")
tokenizer.save_pretrained("triage-lora")
print("Saved LoRA adapter (~50 MB). The 6 GB base model is not saved - just the tiny patch.")


**This is a huge deal.** Full fine-tuning saves a 6 GB file per experiment. LoRA saves ~50 MB. You can keep dozens of adapters and swap them at load time.


## Recap

- **LoRA** = trainable low-rank patches on a frozen base model. ~1% params, ~same quality on style tasks.
- **QLoRA** = LoRA + 4-bit quantized base. Runs on a free Colab GPU.
- **Unsloth** is the recommended 2026 wrapper — 2× faster than raw HF.
- Use the model's chat template — never hand-format tokens.
- Adapters are tiny (~50 MB) and separately loadable.
- **Next class:** the full instruction-tuning workflow on real data.
